<a href="https://colab.research.google.com/github/duruamobi/AAI2026/blob/main/ReACT_Prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =========================
# REASON / PLAN (ReACT Step 1)
# =========================
# Approach:
# - Deterministic keyword matching (no ML, no APIs).
# - Clear priority rule when multiple intents appear:
#     product support > shipping > payment > general inquiry
# - Edge cases:
#     None / empty / whitespace -> general inquiry
#     Unclear messages -> general inquiry
# - Allowed import: re only.

import re

# =========================
# ACT (ReACT Step 2)
# =========================
def classify_ticket(message: str) -> str:
    if message is None:
        return "general inquiry"

    msg = message.strip()
    if not msg:
        return "general inquiry"

    m = msg.lower()

    payment_kw = [
        "payment", "pay", "paid", "charge", "charged", "billing", "invoice",
        "receipt", "refund", "return", "money back", "cancel order",
        "card", "credit card", "debit", "declined", "failed",
        "transaction", "promo", "coupon", "discount", "tax"
    ]

    shipping_kw = [
        "shipping", "ship", "shipment", "tracking", "track", "delivery",
        "delivered", "arrive", "arrival", "eta", "where is", "where's",
        "in transit", "out for delivery", "lost package",
        "missing package", "hasn't arrived", "hasnt arrived"
    ]

    product_support_kw = [
        "broken", "damaged", "defect", "defective",
        "doesn't work", "doesnt work", "not working",
        "won't", "wont", "error", "issue", "problem",
        "malfunction", "wrong item", "wrong size",
        "missing parts", "setup", "assembly",
        "instructions", "warranty", "replace", "replacement"
    ]

    def has_any(keywords):
        return any(k in m for k in keywords)

    # Override rule: refund + shipping delay → shipping
    if ("refund" in m or "money back" in m) and has_any(shipping_kw):
        return "shipping"

    # Priority order
    if has_any(product_support_kw):
        return "product support"
    if has_any(shipping_kw):
        return "shipping"
    if has_any(payment_kw):
        return "payment"

    if re.search(r"(?:order\s*#?|#)\s*[a-z0-9]{5,}", m):
        return "shipping"

    return "general inquiry"


# =========================
# RUN / OBSERVE (ReACT Step 3)
# =========================
tests = [
    "",
    "Hi there",
    "My card was charged twice for my basketball order.",
    "Where is my package? Tracking hasn't updated.",
    "The treadmill arrived damaged and won't turn on.",
    "Do you offer discounts for teams?",
    "Order #A7X92 says delivered but I don't have it.",
    "How do I assemble the weight bench? Missing bolts.",
    "I need a refund but also my package hasn't arrived."
]

for t in tests:
    print(classify_ticket(t))


# =========================
# FIX / ITERATE (ReACT Step 4)
# =========================
# Improvement:
# - Added additional shipping phrase "delayed"
# - Strengthened refund + delay override logic

def classify_ticket_v2(message: str) -> str:
    if message is None:
        return "general inquiry"

    msg = message.strip()
    if not msg:
        return "general inquiry"

    m = msg.lower()

    payment_kw = [
        "payment", "pay", "paid", "charge", "charged", "billing", "invoice",
        "receipt", "refund", "return", "money back", "cancel order",
        "card", "credit card", "debit", "declined", "failed",
        "transaction", "promo", "coupon", "discount", "tax"
    ]

    shipping_kw = [
        "shipping", "ship", "shipment", "tracking", "track", "delivery",
        "delivered", "arrive", "arrival", "eta", "where is", "where's",
        "in transit", "out for delivery", "lost package",
        "missing package", "hasn't arrived", "hasnt arrived",
        "delayed"
    ]

    product_support_kw = [
        "broken", "damaged", "defect", "defective",
        "doesn't work", "doesnt work", "not working",
        "won't", "wont", "error", "issue", "problem",
        "malfunction", "wrong item", "wrong size",
        "missing parts", "setup", "assembly",
        "instructions", "warranty", "replace", "replacement"
    ]

    def has_any(keywords):
        return any(k in m for k in keywords)

    if ("refund" in m or "money back" in m) and has_any(shipping_kw):
        return "shipping"

    if has_any(product_support_kw):
        return "product support"
    if has_any(shipping_kw):
        return "shipping"
    if has_any(payment_kw):
        return "payment"

    if re.search(r"(?:order\s*#?|#)\s*[a-z0-9]{5,}", m):
        return "shipping"

    return "general inquiry"


for t in tests:
    print(classify_ticket_v2(t))

general inquiry
general inquiry
payment
shipping
product support
payment
shipping
general inquiry
shipping
general inquiry
general inquiry
payment
shipping
product support
payment
shipping
general inquiry
shipping
